# Site 100008: parameter and alarm correlation analysis

**Site:** Site_08

This notebook imports the alarm-enhanced six-month site CSV, aligns exact alarm starts to the nearest telemetry timestamp, and investigates which available VRM parameters are associated with alarms.

The analysis distinguishes:

- sampled alarm-active periods from exact alarm starts;
- values during alarms from values 15 and 60 minutes before alarm onset;
- correlation from causation;
- useful candidate thresholds from rules proven to trigger equipment alarms.

Because the telemetry is normally sampled every 15 minutes, short alarms may occur between measurements. Exact alarm starts are therefore mapped to the nearest telemetry row within 20 minutes.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

SITE_ID = 100008
SITE_NAME = 'Site_08'


def find_workspace():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not locate selected_sites_six_month_eda")


WORKSPACE = find_workspace()
SITE_CSV = (
    WORKSPACE
    / "data" / "processed"
    / f"site_{SITE_ID}_six_month_vrm_with_alarms.csv"
)
ALARM_LOG_CSV = WORKSPACE / "data" / "raw" / "vrm_alarm_logs_clean.csv"
RESULT_DIR = WORKSPACE / "results" / f"site_{SITE_ID}"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Site CSV:", SITE_CSV)
print("Alarm log:", ALARM_LOG_CSV)
print("Results:", RESULT_DIR)


## Load and prepare telemetry

Only numeric parameter columns containing observations are analyzed. Alarm metadata columns are excluded from the parameter list.


In [ ]:
df = pd.read_csv(SITE_CSV)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
df.dropna(subset=["timestamp_utc"], inplace=True)
df.sort_values("timestamp_utc", inplace=True)
df.reset_index(drop=True, inplace=True)

df["alarm_active"] = pd.to_numeric(df["alarm_active"], errors="coerce").fillna(0).astype(int)
df["alarm_overlap_count"] = (
    pd.to_numeric(df["alarm_overlap_count"], errors="coerce").fillna(0).astype(int)
)
df["sampled_alarm_onset"] = (
    df["alarm_active"].eq(1)
    & df["alarm_active"].shift(fill_value=0).eq(0)
).astype(int)

excluded_columns = {"site_id", "timestamp_utc", "sampled_alarm_onset"}
metric_columns = []
for column in df.columns:
    if column in excluded_columns or column.startswith("alarm_"):
        continue
    numeric = pd.to_numeric(df[column], errors="coerce")
    if numeric.notna().any():
        df[column] = numeric
        metric_columns.append(column)

coverage = pd.DataFrame({
    "parameter": metric_columns,
    "non_null_rows": [df[column].notna().sum() for column in metric_columns],
    "coverage_pct": [df[column].notna().mean() * 100 for column in metric_columns],
    "unique_values": [df[column].nunique(dropna=True) for column in metric_columns],
})

print("Rows:", len(df))
print("Range:", df["timestamp_utc"].min(), "to", df["timestamp_utc"].max())
print("Alarm-active sampled rows:", int(df["alarm_active"].sum()))
print("Sampled alarm onsets:", int(df["sampled_alarm_onset"].sum()))
display(coverage.round(2))


## Alarm inventory and exact onset alignment

The alarm-enhanced CSV describes intervals that overlap telemetry timestamps. The original alarm log supplies exact start times, descriptions, devices, and classifications. Exact events are mapped to the nearest telemetry row and to telemetry approximately 15 and 60 minutes before onset.


In [ ]:
alarm_log = pd.read_csv(ALARM_LOG_CSV)
site_alarms = alarm_log[alarm_log["site_id"] == SITE_ID].copy()
site_alarms["alarm_started_at"] = pd.to_datetime(
    site_alarms["alarm_started_at"], utc=True, errors="coerce"
)
site_alarms["alarm_cleared_at"] = pd.to_datetime(
    site_alarms["alarm_cleared_at"], utc=True, errors="coerce"
)
site_alarms.dropna(subset=["alarm_started_at"], inplace=True)
site_alarms = site_alarms[
    site_alarms["alarm_started_at"].between(
        df["timestamp_utc"].min(), df["timestamp_utc"].max(), inclusive="both"
    )
].sort_values("alarm_started_at")
site_alarms.reset_index(drop=True, inplace=True)
site_alarms["event_id"] = np.arange(len(site_alarms))

telemetry = df[["timestamp_utc", *metric_columns]].sort_values("timestamp_utc").copy()
telemetry.rename(columns={"timestamp_utc": "matched_timestamp"}, inplace=True)

event_features = pd.merge_asof(
    site_alarms.sort_values("alarm_started_at"),
    telemetry,
    left_on="alarm_started_at",
    right_on="matched_timestamp",
    direction="nearest",
    tolerance=pd.Timedelta(minutes=20),
)
event_features["match_offset_minutes"] = (
    event_features["matched_timestamp"] - event_features["alarm_started_at"]
).dt.total_seconds() / 60


def add_pre_alarm_snapshot(events, minutes):
    targets = events[["event_id", "alarm_started_at"]].copy()
    targets["target_timestamp"] = targets["alarm_started_at"] - pd.Timedelta(minutes=minutes)
    matched = pd.merge_asof(
        targets.sort_values("target_timestamp"),
        telemetry,
        left_on="target_timestamp",
        right_on="matched_timestamp",
        direction="nearest",
        tolerance=pd.Timedelta(minutes=10),
    ).sort_values("event_id")
    matched.set_index("event_id", inplace=True)
    for parameter in metric_columns:
        events[f"pre_{minutes}m__{parameter}"] = events["event_id"].map(matched[parameter])
        events[f"delta_{minutes}m__{parameter}"] = (
            events[parameter] - events[f"pre_{minutes}m__{parameter}"]
        )


add_pre_alarm_snapshot(event_features, 15)
add_pre_alarm_snapshot(event_features, 60)

alarm_inventory = (
    site_alarms.groupby(["classification", "description", "device"], dropna=False)
    .agg(
        alarm_events=("idAlarm", "count"),
        first_alarm=("alarm_started_at", "min"),
        last_alarm=("alarm_started_at", "max"),
        median_duration_minutes=("duration_minutes", "median"),
    )
    .reset_index()
    .sort_values("alarm_events", ascending=False)
)

print("Exact alarms in telemetry range:", len(site_alarms))
print("Alarms mapped to telemetry:", int(event_features["matched_timestamp"].notna().sum()))
display(alarm_inventory.head(20))


## Parameter association with sampled alarm-active periods

`point_biserial_r` is the ordinary correlation between a numeric parameter and the binary alarm-active flag. `median_shift_iqr` compares active and inactive medians relative to the inactive interquartile range. Large absolute values indicate separation, but persistence, seasonality, missing data, and alarm duration can inflate an association.


In [ ]:
def safe_point_biserial(values, indicator):
    paired = pd.DataFrame({"x": values, "y": indicator}).dropna()
    if len(paired) < 3 or paired["x"].nunique() < 2 or paired["y"].nunique() < 2:
        return np.nan
    return paired["x"].corr(paired["y"])


association_rows = []
for parameter in metric_columns:
    valid = df[[parameter, "alarm_active", "sampled_alarm_onset"]].dropna(subset=[parameter])
    inactive = valid.loc[valid["alarm_active"].eq(0), parameter]
    active = valid.loc[valid["alarm_active"].eq(1), parameter]
    inactive_iqr = inactive.quantile(0.75) - inactive.quantile(0.25) if len(inactive) else np.nan
    median_shift_iqr = (
        (active.median() - inactive.median()) / inactive_iqr
        if len(active) and len(inactive) and pd.notna(inactive_iqr) and inactive_iqr != 0
        else np.nan
    )
    association_rows.append({
        "parameter": parameter,
        "observed_rows": len(valid),
        "inactive_rows": len(inactive),
        "active_rows": len(active),
        "inactive_median": inactive.median(),
        "active_median": active.median(),
        "active_minus_inactive_median": active.median() - inactive.median(),
        "median_shift_iqr": median_shift_iqr,
        "point_biserial_r": safe_point_biserial(valid[parameter], valid["alarm_active"]),
        "onset_point_biserial_r": safe_point_biserial(
            valid[parameter], valid["sampled_alarm_onset"]
        ),
    })

parameter_associations = pd.DataFrame(association_rows)
parameter_associations["association_strength"] = (
    parameter_associations["point_biserial_r"].abs()
)
parameter_associations.sort_values(
    ["association_strength", "active_rows"], ascending=[False, False], inplace=True
)
display(parameter_associations.round(4))


## Alarm rate by parameter value range

Each parameter is divided into up to ten quantile bins. `alarm_rate_lift` is the bin's sampled alarm-active rate divided by the parameter's overall alarm-active rate. A lift above 1 means alarms were more common in that value range. Bins with few observations should not be treated as thresholds.


In [ ]:
risk_bin_parts = []
for parameter in metric_columns:
    valid = df[[parameter, "alarm_active", "sampled_alarm_onset"]].dropna().copy()
    if len(valid) < 30 or valid[parameter].nunique() < 3:
        continue
    try:
        valid["value_bin"] = pd.qcut(valid[parameter], q=10, duplicates="drop")
    except ValueError:
        continue

    overall_active_rate = valid["alarm_active"].mean()
    overall_onset_rate = valid["sampled_alarm_onset"].mean()
    grouped = (
        valid.groupby("value_bin", observed=True)
        .agg(
            bin_min=(parameter, "min"),
            bin_max=(parameter, "max"),
            bin_median=(parameter, "median"),
            rows=(parameter, "size"),
            active_rows=("alarm_active", "sum"),
            alarm_rate=("alarm_active", "mean"),
            onset_rows=("sampled_alarm_onset", "sum"),
            onset_rate=("sampled_alarm_onset", "mean"),
        )
        .reset_index(drop=True)
    )
    grouped.insert(0, "parameter", parameter)
    grouped["alarm_rate_lift"] = (
        grouped["alarm_rate"] / overall_active_rate if overall_active_rate > 0 else np.nan
    )
    grouped["onset_rate_lift"] = (
        grouped["onset_rate"] / overall_onset_rate if overall_onset_rate > 0 else np.nan
    )
    risk_bin_parts.append(grouped)

parameter_risk_bins = (
    pd.concat(risk_bin_parts, ignore_index=True)
    if risk_bin_parts
    else pd.DataFrame()
)
display(
    parameter_risk_bins.sort_values("alarm_rate_lift", ascending=False).head(30).round(4)
    if not parameter_risk_bins.empty
    else pd.DataFrame({"note": ["No risk bins could be calculated."]})
)


## Values observed at exact alarm starts

This table compares exact-alarm values with the non-alarm baseline. It also reports how often alarm-start values are below the baseline 10th percentile or above the baseline 90th percentile, and the median change from 15 and 60 minutes before onset.

Candidate thresholds are exploratory. The alarm log does not state the manufacturer's internal trigger limits, and some alarms are unrelated to energy-flow parameters.


In [ ]:
onset_rows = []
for parameter in metric_columns:
    baseline = df.loc[df["alarm_active"].eq(0), parameter].dropna()
    alarm_values = event_features[parameter].dropna()
    if baseline.empty:
        continue

    baseline_p10 = baseline.quantile(0.10)
    baseline_p50 = baseline.quantile(0.50)
    baseline_p90 = baseline.quantile(0.90)
    baseline_iqr = baseline.quantile(0.75) - baseline.quantile(0.25)
    event_median = alarm_values.median() if len(alarm_values) else np.nan
    standardized_event_shift = (
        (event_median - baseline_p50) / baseline_iqr
        if len(alarm_values) and pd.notna(baseline_iqr) and baseline_iqr != 0
        else np.nan
    )
    below_p10_pct = (
        (alarm_values < baseline_p10).mean() * 100 if len(alarm_values) else np.nan
    )
    above_p90_pct = (
        (alarm_values > baseline_p90).mean() * 100 if len(alarm_values) else np.nan
    )

    if len(alarm_values) < 10:
        direction = "insufficient alarm-start values (<10)"
        threshold = np.nan
    elif pd.isna(below_p10_pct) or pd.isna(above_p90_pct):
        direction = "insufficient alarm-start values"
        threshold = np.nan
    elif above_p90_pct > below_p10_pct:
        direction = "high values associated with alarm starts"
        threshold = baseline_p90
    elif below_p10_pct > above_p90_pct:
        direction = "low values associated with alarm starts"
        threshold = baseline_p10
    else:
        direction = "no clear high/low direction"
        threshold = np.nan

    onset_rows.append({
        "parameter": parameter,
        "mapped_alarm_values": len(alarm_values),
        "baseline_rows": len(baseline),
        "baseline_p10": baseline_p10,
        "baseline_median": baseline_p50,
        "baseline_p90": baseline_p90,
        "alarm_start_median": event_median,
        "standardized_event_shift_iqr": standardized_event_shift,
        "alarm_values_below_baseline_p10_pct": below_p10_pct,
        "alarm_values_above_baseline_p90_pct": above_p90_pct,
        "median_delta_15m": event_features[f"delta_15m__{parameter}"].median(),
        "median_delta_60m": event_features[f"delta_60m__{parameter}"].median(),
        "candidate_direction": direction,
        "candidate_threshold": threshold,
    })

onset_findings = pd.DataFrame(onset_rows)
if not onset_findings.empty:
    capped_shift = onset_findings["standardized_event_shift_iqr"].abs().fillna(0).clip(
        upper=5
    )
    extreme_fraction = (
        onset_findings[
            [
                "alarm_values_below_baseline_p10_pct",
                "alarm_values_above_baseline_p90_pct",
            ]
        ]
        .max(axis=1)
        .fillna(0)
        / 100
    )
    onset_findings["evidence_score"] = (
        capped_shift
        * np.log1p(onset_findings["mapped_alarm_values"])
        * extreme_fraction
    )
    onset_findings.loc[
        onset_findings["mapped_alarm_values"] < 10, "evidence_score"
    ] = 0
    onset_findings.loc[
        onset_findings["candidate_threshold"].isna(), "evidence_score"
    ] = 0
    onset_findings.sort_values("evidence_score", ascending=False, inplace=True)

display(onset_findings.round(4))


## Parameter patterns by alarm classification

Different alarm classes may have different parameter signatures. Results are produced only when at least three exact events in a class have a mapped value for the parameter.


In [ ]:
alarm_type_rows = []
if not event_features.empty and "classification" in event_features:
    for classification, group in event_features.groupby("classification", dropna=False):
        for parameter in metric_columns:
            values = group[parameter].dropna()
            baseline = df.loc[df["alarm_active"].eq(0), parameter].dropna()
            if len(values) < 3 or baseline.empty:
                continue
            p10 = baseline.quantile(0.10)
            p90 = baseline.quantile(0.90)
            alarm_type_rows.append({
                "classification": classification,
                "parameter": parameter,
                "mapped_events": len(values),
                "event_median": values.median(),
                "baseline_median": baseline.median(),
                "event_below_baseline_p10_pct": (values < p10).mean() * 100,
                "event_above_baseline_p90_pct": (values > p90).mean() * 100,
                "median_delta_15m": group[f"delta_15m__{parameter}"].median(),
                "median_delta_60m": group[f"delta_60m__{parameter}"].median(),
            })

alarm_type_findings = pd.DataFrame(alarm_type_rows)
if not alarm_type_findings.empty:
    alarm_type_findings["extreme_event_pct"] = alarm_type_findings[
        ["event_below_baseline_p10_pct", "event_above_baseline_p90_pct"]
    ].max(axis=1)
    alarm_type_findings.sort_values(
        ["extreme_event_pct", "mapped_events"], ascending=[False, False], inplace=True
    )

display(
    alarm_type_findings.head(50).round(4)
    if not alarm_type_findings.empty
    else pd.DataFrame({"note": ["No alarm class has enough mapped observations."]})
)


## Graphs

The following figures map available parameters to alarm periods and alarm starts:

1. normalized parameter time series with alarm activity;
2. point-biserial association strength;
3. inactive-versus-active parameter distributions;
4. alarm-rate lift by parameter quantile;
5. alarm classification frequency;
6. alarm-start parameter percentile relative to baseline.


In [ ]:
plot_metrics = (
    parameter_associations.dropna(subset=["association_strength"])
    .head(4)["parameter"]
    .tolist()
)
if not plot_metrics:
    plot_metrics = metric_columns[:4]


def save_and_show(fig, filename):
    fig.tight_layout()
    fig.savefig(RESULT_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# 1. Robust-normalized time series with alarm activity.
fig, axis = plt.subplots(figsize=(16, 7))
for parameter in plot_metrics:
    series = df.set_index("timestamp_utc")[parameter]
    median = series.median()
    iqr = series.quantile(0.75) - series.quantile(0.25)
    normalized = (series - median) / iqr if pd.notna(iqr) and iqr != 0 else series * np.nan
    axis.plot(normalized.index, normalized, linewidth=0.7, label=parameter)
alarm_mask = df.set_index("timestamp_utc")["alarm_active"].astype(bool)
axis.fill_between(
    alarm_mask.index,
    -4,
    4,
    where=alarm_mask,
    color="crimson",
    alpha=0.12,
    label="alarm active",
)
axis.set_ylim(-4, 4)
axis.set_title(f"Site {SITE_ID}: normalized parameters and sampled alarm periods")
axis.set_ylabel("Robust normalized value (median / IQR)")
axis.grid(True, alpha=0.2)
axis.legend(loc="upper right", ncol=2)
save_and_show(fig, "01_parameter_timeline_with_alarms.png")


# 2. Point-biserial association.
fig, axis = plt.subplots(figsize=(11, max(4, len(parameter_associations) * 0.45)))
plot_association = parameter_associations.sort_values("point_biserial_r")
colors = ["#b91c1c" if value < 0 else "#0369a1" for value in plot_association["point_biserial_r"].fillna(0)]
axis.barh(plot_association["parameter"], plot_association["point_biserial_r"], color=colors)
axis.axvline(0, color="black", linewidth=0.8)
axis.set_title("Parameter correlation with sampled alarm-active status")
axis.set_xlabel("Point-biserial correlation")
axis.grid(True, axis="x", alpha=0.2)
save_and_show(fig, "02_parameter_alarm_correlation.png")


# 3. Distribution comparison.
fig, axes = plt.subplots(len(plot_metrics), 1, figsize=(11, 3.2 * len(plot_metrics)))
if len(plot_metrics) == 1:
    axes = [axes]
for axis, parameter in zip(axes, plot_metrics):
    inactive = df.loc[df["alarm_active"].eq(0), parameter].dropna()
    active = df.loc[df["alarm_active"].eq(1), parameter].dropna()
    data = [inactive, active] if len(active) else [inactive, pd.Series(dtype=float)]
    axis.boxplot(data, labels=["inactive", "active"], showfliers=False)
    combined = pd.concat([inactive, active])
    if len(combined):
        low, high = combined.quantile([0.01, 0.99])
        if pd.notna(low) and pd.notna(high) and low < high:
            axis.set_ylim(low, high)
    axis.set_title(parameter)
    axis.grid(True, axis="y", alpha=0.2)
fig.suptitle(f"Site {SITE_ID}: parameter distributions during inactive and active alarms", y=1.01)
save_and_show(fig, "03_parameter_distributions_by_alarm_status.png")


# 4. Alarm-rate lift by quantile.
fig, axes = plt.subplots(len(plot_metrics), 1, figsize=(11, 3.2 * len(plot_metrics)))
if len(plot_metrics) == 1:
    axes = [axes]
for axis, parameter in zip(axes, plot_metrics):
    bins = parameter_risk_bins[parameter_risk_bins["parameter"] == parameter]
    if bins.empty:
        axis.text(0.5, 0.5, "Insufficient data", ha="center", va="center")
    else:
        axis.plot(
            np.arange(1, len(bins) + 1),
            bins["alarm_rate_lift"],
            marker="o",
            color="#7c3aed",
        )
        axis.axhline(1, color="black", linestyle="--", linewidth=0.8)
        axis.set_xticks(np.arange(1, len(bins) + 1))
        axis.set_xlabel("Value quantile: low to high")
        axis.set_ylabel("Alarm-rate lift")
    axis.set_title(parameter)
    axis.grid(True, alpha=0.2)
fig.suptitle(f"Site {SITE_ID}: alarm frequency across parameter value ranges", y=1.01)
save_and_show(fig, "04_alarm_rate_lift_by_parameter_range.png")


# 5. Alarm classification frequency.
fig, axis = plt.subplots(figsize=(12, 6))
if alarm_inventory.empty:
    axis.text(0.5, 0.5, "No exact alarms in the telemetry range", ha="center", va="center")
    axis.set_axis_off()
else:
    class_counts = (
        site_alarms["classification"].fillna("unclassified").value_counts().head(15).sort_values()
    )
    axis.barh(class_counts.index.astype(str), class_counts.values, color="#ea580c")
    axis.set_xlabel("Alarm events")
    axis.set_title("Most frequent alarm classifications")
    axis.grid(True, axis="x", alpha=0.2)
save_and_show(fig, "05_alarm_classification_counts.png")


# 6. Alarm-start percentile relative to baseline.
fig, axis = plt.subplots(figsize=(11, max(4, len(onset_findings) * 0.5)))
if onset_findings.empty:
    axis.text(0.5, 0.5, "No mapped alarm-start parameter values", ha="center", va="center")
    axis.set_axis_off()
else:
    percentile_plot = onset_findings[
        onset_findings["mapped_alarm_values"] >= 10
    ].sort_values("evidence_score")
    if percentile_plot.empty:
        axis.text(0.5, 0.5, "Fewer than 10 mapped values per parameter", ha="center", va="center")
        axis.set_axis_off()
        percentile_plot = None
if not onset_findings.empty and percentile_plot is not None:
    y = np.arange(len(percentile_plot))
    axis.barh(
        y - 0.18,
        percentile_plot["alarm_values_below_baseline_p10_pct"],
        height=0.35,
        label="below baseline P10",
        color="#2563eb",
    )
    axis.barh(
        y + 0.18,
        percentile_plot["alarm_values_above_baseline_p90_pct"],
        height=0.35,
        label="above baseline P90",
        color="#dc2626",
    )
    axis.set_yticks(y)
    axis.set_yticklabels(percentile_plot["parameter"])
    axis.set_xlabel("Alarm-start observations (%)")
    axis.set_title("Extreme parameter values at exact alarm starts")
    axis.legend()
    axis.grid(True, axis="x", alpha=0.2)
save_and_show(fig, "06_extreme_values_at_alarm_start.png")


## Export tables and review candidate findings

The exported findings provide candidate value ranges for engineering review. They should be validated against alarm definitions, device-specific limits, and additional operational data before being used as alert rules.


In [ ]:
coverage.to_csv(RESULT_DIR / "parameter_coverage.csv", index=False)
alarm_inventory.to_csv(RESULT_DIR / "alarm_inventory.csv", index=False)
event_features.to_csv(RESULT_DIR / "exact_alarm_event_features.csv", index=False)
parameter_associations.to_csv(
    RESULT_DIR / "sampled_parameter_alarm_associations.csv", index=False
)
parameter_risk_bins.to_csv(RESULT_DIR / "parameter_alarm_risk_bins.csv", index=False)
onset_findings.to_csv(RESULT_DIR / "parameter_alarm_onset_findings.csv", index=False)
alarm_type_findings.to_csv(
    RESULT_DIR / "alarm_type_parameter_findings.csv", index=False
)

analysis_summary = pd.DataFrame([{
    "site_id": SITE_ID,
    "site_name": SITE_NAME,
    "telemetry_rows": len(df),
    "available_parameters": len(metric_columns),
    "sampled_alarm_active_rows": int(df["alarm_active"].sum()),
    "sampled_alarm_onsets": int(df["sampled_alarm_onset"].sum()),
    "exact_alarms_in_range": len(site_alarms),
    "exact_alarms_mapped_to_telemetry": int(event_features["matched_timestamp"].notna().sum()),
    "strongest_sampled_association_parameter": (
        parameter_associations.iloc[0]["parameter"] if not parameter_associations.empty else None
    ),
    "strongest_sampled_point_biserial_r": (
        parameter_associations.iloc[0]["point_biserial_r"]
        if not parameter_associations.empty
        else np.nan
    ),
    "top_onset_candidate_parameter": (
        onset_findings.loc[onset_findings["evidence_score"] > 0].iloc[0]["parameter"]
        if (not onset_findings.empty and (onset_findings["evidence_score"] > 0).any())
        else None
    ),
    "top_onset_candidate_direction": (
        onset_findings.loc[onset_findings["evidence_score"] > 0].iloc[0]["candidate_direction"]
        if (not onset_findings.empty and (onset_findings["evidence_score"] > 0).any())
        else None
    ),
    "top_onset_candidate_threshold": (
        onset_findings.loc[onset_findings["evidence_score"] > 0].iloc[0]["candidate_threshold"]
        if (not onset_findings.empty and (onset_findings["evidence_score"] > 0).any())
        else np.nan
    ),
}])
analysis_summary.to_csv(RESULT_DIR / "site_analysis_summary.csv", index=False)

print("Saved analysis tables and figures to:", RESULT_DIR)
display(analysis_summary)
display(onset_findings.head(10).round(4))
